# IND320 Project P1

Author: Robin Torjussen Egede  
[Streamlit P2 page](https://ind320-hush-p2.streamlit.app) - [P1 page](https://ind320-hush.streamlit.app)  
[Repo & P2 branch](https://github.com/Kunii/IND320-hush/tree/work-p2)  

## Log

- Created snapshot branch of P1 [(found here)](https://github.com/Kunii/IND320-hush/tree/snapshot-p1).
- Created new work branch for P2.
- Reorganized 'loose' P1 files into 'CA1' dirs.
- Added streamlit link to readme.
- Updated package deps (pyspark, pymongo etc).
- Set up cassandra & verified working ops.
    - Fetched container image via `docker pull cassandra:latest`.
    - Standard docker ops to bring it up, no custom config other than specifying passthrough-port.
- Set up pyspark.
    - Never mind it throws errors all over the place. Py 3.14 / jre issue likely.
- Switched to python 3.12 + updated deps again for pyspark compat.
    - `.python-version` + `pyproject.toml`: 3.14 -> 3.12.
    - `pyproject.toml`: Cleared all strict version tags from deps (Except `pyspark`).
- Installed JDK 17 for pyspark compat.
    - `apt install openjdk-17-jdk` -> Done. Long live linux terminal installing.
    - Env var `JAVA_HOME` set to install path @ runtime.
    - Pyspark works fine now!
- Set up mongodb & verified working ops.
- Initial `Magasinstatistikk` api probe.
    - Py `requests` std lib.
    - Std `get` req works fine, no token required.
- Registered @ entsoe. Guess I wait & pray I get API access before due date :)
- Created auto-runner for the cassandra container via the docker package.

## AI-usage

Used as "fancy auto-complete" in-line suggestions, docstring & in-line comment drafts, plotting drafts.  
Also used for general idea & "best practice" for the secrets stuff.  
All logic and program flow was strictly concocted by me.

In [11]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from cassandra.util import datetime_from_uuid1
from cassandra.cluster import Cluster
from pyspark.sql import SparkSession
from pymongo import MongoClient
from src.utils.env import secrets

import pandas as pd
import matplotlib.pyplot as plt
import requests

from dataclasses import dataclass, field
from typing import Optional, Union, List, Any

from docker import from_env

In [12]:
mongo_username = secrets.get_required("mongo_db.username")
mongo_password = secrets.get_required("mongo_db.password")
mongo_url = secrets.get_required("mongo_db.url").replace("<username>", mongo_username).replace("<password>", mongo_password)

entsoe_token = secrets.get_required("entsoe.token")

In [6]:
# Automated container start
docker_client = from_env()
container = docker_client.containers.get("ind320-cassandra")

if container.status != "running":
    container.start()
    print("Cassandra container started.")
else:
    print("Status:", container.status)

print(container.logs(tail=5).decode("utf-8"))

Status: running
INFO  [IndexSummaryManager:1] 2026-10-19T14:44:22,467 IndexSummaryRedistribution.java:115 - Beginning redistribution of index summaries for 37 sstables with memory pool size 1175 MiB; current spaced used is 9.107589721679688E-4 MiB
INFO  [IndexSummaryManager:1] 2026-10-19T14:44:22,468 IndexSummaryRedistribution.java:268 - index summaries: downsample: 0, force resample: 0, upsample: 0, force upsample: 0
INFO  [IndexSummaryManager:1] 2026-10-19T14:44:22,469 IndexSummaryRedistribution.java:154 - Completed resizing of index summaries; current approximate memory used: 9.107589721679688E-4 MiB, time spent: 2ms
INFO  [PerDiskMemtableFlushWriter_0:44] 2026-10-19T14:44:22,476 Flushing.java:153 - Writing Memtable-local@529737795(65B serialized bytes, 2 ops, 553B (0%) on-heap, 0B (0%) off-heap), flushed range = [min(-9223372036854775808), max(9223372036854775807))
INFO  [PerDiskMemtableFlushWriter_0:44] 2026-10-19T14:44:22,477 Flushing.java:179 - Completed flushing /var/lib/cassan

In [7]:
# Connecting to Cassandra
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

db_keyspace = "ind320"
db_table = "testing_table"

session.execute(f"CREATE KEYSPACE IF NOT EXISTS {db_keyspace} WITH REPLICATION = {{ 'class': 'SimpleStrategy', 'replication_factor': 1 }};")
session.set_keyspace(db_keyspace)

session.execute(f"DROP TABLE IF EXISTS {db_table};")

session.execute(f"""CREATE TABLE IF NOT EXISTS {db_table} (
    id TIMEUUID PRIMARY KEY,
    name TEXT,
    age INT,
    gender TEXT,
    created_at TIMESTAMP
);""")

In [8]:
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'John Doe', 30, 'male', currentTimestamp());")
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'Jane Smith', 25, 'female', currentTimestamp());")
session.execute(f"INSERT INTO {db_table} (id, name, age, gender, created_at) VALUES (now(), 'Robin E', 29, 'male', currentTimestamp());")

In [9]:
# Query the data
rows = session.execute(f"SELECT * FROM {db_table};")
for i in rows:
    print(i)
    # Extract the timestamp from Cassandra's timeuuid
    print("Datetime:", datetime_from_uuid1(i.id))

Row(id=UUID('12440d60-cbcd-11f1-8ee4-ed9664de04bf'), age=29, created_at=datetime.datetime(2026, 10, 19, 14, 55, 6, 38000), gender='male', name='Robin E')
Datetime: 2026-10-19 14:55:06.038000
Row(id=UUID('12439830-cbcd-11f1-8ee4-ed9664de04bf'), age=25, created_at=datetime.datetime(2026, 10, 19, 14, 55, 6, 35000), gender='female', name='Jane Smith')
Datetime: 2026-10-19 14:55:06.035000
Row(id=UUID('12434a10-cbcd-11f1-8ee4-ed9664de04bf'), age=30, created_at=datetime.datetime(2026, 10, 19, 14, 55, 6, 33000), gender='male', name='John Doe')
Datetime: 2026-10-19 14:55:06.033000


In [10]:

spark = (
    SparkSession.builder.appName('SparkCassandraApp')
    .config('spark.jars.packages',
            'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1')
#     .config('spark.sql.extensions',
#             'com.datastax.spark.connector.CassandraSparkExtensions')
#     .config('spark.sql.catalog.mycatalog',
#             'com.datastax.spark.connector.datasource.CassandraCatalog')
    .config('spark.cassandra.connection.host',
            '127.0.0.1')
    .config('spark.cassandra.connection.port',
            '9042')
    .getOrCreate()
)

26/10/19 16:55:06 WARN Utils: Your hostname, hush-kubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.3 instead (on interface wlo1)
26/10/19 16:55:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/nvme_storage/projects/NMBU/IND320-hush/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hush/.ivy2/cache
The jars for the packages stored in: /home/hush/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c5997a6-9f58-42d4-a849-8b954bb36249;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3 i

In [11]:
spark_df = spark.read.format("org.apache.spark.sql.cassandra").options(table=db_table, keyspace=db_keyspace).load()

spark_df.show()

+--------------------+---+--------------------+------+----------+
|                  id|age|          created_at|gender|      name|
+--------------------+---+--------------------+------+----------+
|12440d60-cbcd-11f...| 29|2026-10-19 16:55:...|  male|   Robin E|
|12439830-cbcd-11f...| 25|2026-10-19 16:55:...|female|Jane Smith|
|12434a10-cbcd-11f...| 30|2026-10-19 16:55:...|  male|  John Doe|
+--------------------+---+--------------------+------+----------+



No secrets path provided
Defaulting to: [REDACTED]/IND320-hush/.secrets/secrets.toml


In [13]:
client = MongoClient(mongo_url)
try:
    client.admin.command("ping")
    print("Connected successfully")
    #client.close()

except Exception as e:
    raise Exception(
        "The following error occurred: ", e)

Connected successfully


In [14]:
db = client["ind320"]
collection = db["testing_collection"]

collection.drop() # Drop the collection if it exists to start fresh

collection.insert_one({"name": "John Doe", "age": 30, "gender": "male"})
collection.insert_many([
    {"name": "Jane Smith", "age": 25, "gender": "female"},
    {"name": "Robin E", "age": 29, "gender": "male"}
])

for row in collection.find():
    print(row)

client.close()

{'_id': ObjectId('6ad62f508a092bc03d9bd03f'), 'name': 'John Doe', 'age': 30, 'gender': 'male'}
{'_id': ObjectId('6ad62f508a092bc03d9bd040'), 'name': 'Jane Smith', 'age': 25, 'gender': 'female'}
{'_id': ObjectId('6ad62f508a092bc03d9bd041'), 'name': 'Robin E', 'age': 29, 'gender': 'male'}


In [15]:
API = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

# init data
req_json = requests.get(API).json()
df_mag = pd.DataFrame(req_json)

df_mag = df_mag.rename(columns={
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "water_level",
    "kapasitet_TWh": "twh_cap",
    "fylling_TWh": "twh_fill",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "water_level_prev_week",
    "endring_fyllingsgrad": "water_level_change"
})

display(df_mag.head())

,date,area_type,area_number,iso_year,iso_week,water_level,twh_cap,twh_fill,next_publication_date,water_level_prev_week,water_level_change
0,1995-09-03,EL,4,1995,35,0.944721,21.079208,19.913979,0001-01-01T00:00:00,0.936888,0.007833
1,2020-11-15,EL,1,2020,46,0.974361,6.003264,5.849344,2020-11-25T13:00:00,0.997406,-0.023046
2,2011-08-28,EL,4,2011,34,0.786499,21.079208,16.578772,0001-01-01T00:00:00,0.787193,-0.000694
3,1998-07-05,EL,3,1998,27,0.793969,8.920932,7.082947,0001-01-01T00:00:00,0.733812,0.060157
4,2011-03-27,EL,4,2011,12,0.300820,21.079208,6.341054,0001-01-01T00:00:00,0.311122,-0.010301


In [79]:


@dataclass
class EICAreaYCode:
    
    EIC_CODE: str
    DISPLAY_NAME: str
    NAME: str
    EIC_PARENT: Optional[str] = None
    EIC_RESPONSIBLE: Optional[str] = None
    VAT_CODE: Optional[str] = None
    FUNCTION: List[str] = field(default_factory=list)

    @classmethod
    def from_raw(
        cls,
        eic_code: Any | str,
        display_name: Any | str,
        name: Any | str,
        eic_parent: Optional[str] = None,
        eic_responsible: Optional[str] = None,
        vat_code: Optional[str] = None,
        function: Union[Any, str, List[str]] = "",
    ) -> "EICAreaYCode":
        
        func_list = [f.strip() for f in function.split(",") if f.strip() and f.strip().lower() != "none"] if isinstance(function, str) else function or []

        def clean_val(val):
            if pd.isna(val) or val is None or str(val).strip().lower() in ("none", "", "nan"):
                return None
            return str(val).strip()

        return cls(
            EIC_CODE=str(eic_code).strip(),
            DISPLAY_NAME=str(display_name).strip() if pd.notna(display_name) else "",
            NAME=str(name).strip() if pd.notna(name) else "",
            EIC_PARENT=clean_val(eic_parent),
            EIC_RESPONSIBLE=clean_val(eic_responsible),
            VAT_CODE=clean_val(vat_code),
            FUNCTION=func_list
        )

def load_eic_codes(filepath: str) -> List[EICAreaYCode]:
    df = pd.read_csv(filepath, sep=';', encoding='utf-8-sig')
    return [
        EICAreaYCode.from_raw(
            eic_code=row.get('EicCode'),
            display_name=row.get('EicDisplayName'),
            name=row.get('EicLongName'),
            eic_parent=row.get('EicParent'),
            eic_responsible=row.get('EicResponsibleParty'),
            vat_code=row.get('MarketParticipantVatCode'),
            function=row.get('EicTypeFunctionList')
        )
        for _, row in df.iterrows()
    ]


eic_objects = load_eic_codes('/mnt/nvme_storage/projects/NMBU/IND320-hush/data/Y_eicCodes.csv')
print(f"Loaded {len(eic_objects)} records.")
print("Sample:", eic_objects[0])

# Print all norwegian EIC codes
norwegian_eic_codes = [eic for eic in eic_objects if eic.EIC_CODE.startswith("10YNO")]
all_eic_countries = [eic for eic in eic_objects if eic.EIC_CODE.startswith("10Y") and len(eic.DISPLAY_NAME) == 2]
eic_no_borders = [eic for eic in all_eic_countries if eic.DISPLAY_NAME in ['FI', 'NL', 'SE', 'DE', 'DK', 'GB']]

Loaded 1829 records.
Sample: EICAreaYCode(EIC_CODE='10Y1001A1001A008', DISPLAY_NAME='BE-FR-NL-MARKET', NAME='Market coupling area Belgium - France - Netherlands', EIC_PARENT=None, EIC_RESPONSIBLE=None, VAT_CODE=None, FUNCTION=['Scheduling Area'])


In [27]:
from xml.etree import ElementTree as ET

In [115]:
entsoe_url = "https://web-api.tp.entsoe.eu/api?"

end = pd.Timestamp.today()

# Offset start date by 2 years
start = end - pd.DateOffset(years=2)


def get_entsoe_data(start, end, eic_in, eic_out) -> requests.Response | List[requests.Response]:
    
    # If start, end interval is more than 1 year, split into multiple requests
    if (end - start).days > 365:
        
        responses = []
        current_end = start  # Required initialization
        
        while current_end < end:
            current_start = current_end
            current_end = min(current_start + pd.DateOffset(years=1), end)
            
            responses.append(get_entsoe_data(current_start, current_end, eic_in, eic_out))
            
        return responses
    
    req_params = {
        "securityToken":    entsoe_token,                   # Private security token for api access
        "documentType":     "A11",                          # A11 = Aggregated energy data report
        "out_Domain":       eic_out,                        # EIC code of a Border
        "in_Domain":        eic_in,                         # EIC code of a Border
        "periodStart":      start.strftime("%Y%m%d%H%M"),   # Pattern yyyyMMddHHmm
        "periodEnd":        end.strftime("%Y%m%d%H%M"),     # Pattern yyyyMMddHHmm
    }
    
    print(f"Fetching data from {start} to {end} for EIC codes: in={eic_in}, out={eic_out}")
    
    return requests.get(entsoe_url, params=req_params)


In [117]:
data = []
for country in eic_no_borders:
    print(f"Fetching data for {country.DISPLAY_NAME} ({country.EIC_CODE})")
    req = get_entsoe_data(start, end, norwegian_eic_codes[0].EIC_CODE, country.EIC_CODE)
    
    if isinstance(req, list):
        trees = [ET.fromstring(r.text) for r in req]
        base_tree = trees[0]
        for tree in trees[1:]:
            for child in tree:
                if child.tag.endswith('TimeSeries'):
                    base_tree.append(child)
        xml = base_tree
    else:
        xml = ET.fromstring(req.text)

    data.append((country.DISPLAY_NAME, country.EIC_CODE, xml))

Fetching data for DK (10Y1001A1001A65H)
Fetching data from 2024-10-19 20:30:41.148872 to 2025-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10Y1001A1001A65H
Fetching data from 2025-10-19 20:30:41.148872 to 2026-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10Y1001A1001A65H
Fetching data for DE (10Y1001A1001A83F)
Fetching data from 2024-10-19 20:30:41.148872 to 2025-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10Y1001A1001A83F
Fetching data from 2025-10-19 20:30:41.148872 to 2026-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10Y1001A1001A83F
Fetching data for FI (10YFI-1--------U)
Fetching data from 2024-10-19 20:30:41.148872 to 2025-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10YFI-1--------U
Fetching data from 2025-10-19 20:30:41.148872 to 2026-10-19 20:30:41.148872 for EIC codes: in=10YNO-0--------C, out=10YFI-1--------U
Fetching data for GB (10YGB----------A)
Fetching data from 2024-10-19 20:30:41.148

In [126]:
#for i, (country_name, country_code, xml) in enumerate(data):
#    print(f"Saving data for {country_name} ({country_code})")
#    with open(f"/mnt/nvme_storage/projects/NMBU/IND320-hush/data/entsoe_data_{country_code}_{i}.xml", "wb") as f:
#        f.write(ET.tostring(xml, encoding='utf-8', xml_declaration=True))

rows = []

for _, _, xml in data:
    # xml is already an Element (root), not an ElementTree
    root = xml
    
    for ts in root.findall(".//{*}TimeSeries"):
        business_type = ts.find("{*}businessType").text
        in_domain_elem = ts.find("{*}in_Domain.mRID").text
        out_domain_elem = ts.find("{*}out_Domain.mRID").text
        
        unit_elem = ts.find("{*}quantity_Measure_Unit.name")
        unit = unit_elem.text if unit_elem is not None else None
        
        for period in ts.findall("{*}Period"):
            start_elem = period.find("{*}timeInterval/{*}start")
            resolution_elem = period.find("{*}resolution")
            
            if start_elem is None or resolution_elem is None:
                continue
            
            period_start = pd.to_datetime(start_elem.text)
            resolution = resolution_elem.text
            
            if resolution == "PT60M":
                freq = pd.Timedelta(hours=1)
            elif resolution == "PT15M":
                freq = pd.Timedelta(minutes=15)
            elif resolution == "P1D":
                freq = pd.Timedelta(days=1)
            else:
                freq = pd.Timedelta(resolution)
            
            for point in period.findall("{*}Point"):
                pos_elem = point.find("{*}position")
                qty_elem = point.find("{*}quantity")
                
                if pos_elem is None or qty_elem is None:
                    continue
                
                position = int(pos_elem.text)
                quantity = float(qty_elem.text)
                timestamp = period_start + freq * (position - 1)
                
                rows.append({
                    "in_domain": in_domain_elem,
                    "out_domain": out_domain_elem,
                    "timestamp": timestamp,
                    "quantity": quantity,
                    "unit": unit,
                    "resolution": resolution,
                    "business_type": business_type
                })
                
df = pd.DataFrame(rows)

if not df.empty:
    df = df.sort_values(["out_domain", "timestamp"]).reset_index(drop=True)

df

,in_domain,out_domain,timestamp,quantity,unit,resolution,business_type
0,10YNO-0--------C,10Y1001A1001A65H,2024-10-19 20:00:00+00:00,0.000000,MAW,PT60M,A66
1,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 08:00:00+00:00,22.410000,MAW,PT60M,A66
2,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 09:00:00+00:00,385.600000,MAW,PT60M,A66
3,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 10:00:00+00:00,664.130000,MAW,PT60M,A66
4,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 11:00:00+00:00,748.760000,MAW,PT60M,A66
...,...,...,...,...,...,...,...
86466,10YNO-0--------C,10YSE-1--------K,2026-10-19 16:45:00+00:00,142.394402,MAW,PT15M,A66
86467,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:00:00+00:00,187.579018,MAW,PT15M,A66
86468,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:15:00+00:00,260.305923,MAW,PT15M,A66
86469,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:30:00+00:00,365.808594,MAW,PT15M,A66


In [82]:
spark = (
    SparkSession.builder.appName('SparkCassandraApp')
    .config('spark.jars.packages',
            'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1')
#     .config('spark.sql.extensions',
#             'com.datastax.spark.connector.CassandraSparkExtensions')
#     .config('spark.sql.catalog.mycatalog',
#             'com.datastax.spark.connector.datasource.CassandraCatalog')
    .config('spark.cassandra.connection.host',
            '127.0.0.1')
    .config('spark.cassandra.connection.port',
            '9042')
    .getOrCreate()
)

26/10/19 19:58:04 WARN Utils: Your hostname, hush-kubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.3 instead (on interface wlo1)
26/10/19 19:58:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/nvme_storage/projects/NMBU/IND320-hush/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hush/.ivy2/cache
The jars for the packages stored in: /home/hush/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-46f4613c-e704-4f2b-95f3-f490c3aca31f;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3 i

In [131]:
display(df)

,in_domain,out_domain,timestamp,quantity,unit,resolution,business_type
0,10YNO-0--------C,10Y1001A1001A65H,2024-10-19 20:00:00+00:00,0.000000,MAW,PT60M,A66
1,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 08:00:00+00:00,22.410000,MAW,PT60M,A66
2,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 09:00:00+00:00,385.600000,MAW,PT60M,A66
3,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 10:00:00+00:00,664.130000,MAW,PT60M,A66
4,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 11:00:00+00:00,748.760000,MAW,PT60M,A66
...,...,...,...,...,...,...,...
86466,10YNO-0--------C,10YSE-1--------K,2026-10-19 16:45:00+00:00,142.394402,MAW,PT15M,A66
86467,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:00:00+00:00,187.579018,MAW,PT15M,A66
86468,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:15:00+00:00,260.305923,MAW,PT15M,A66
86469,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:30:00+00:00,365.808594,MAW,PT15M,A66


In [ ]:
str(df['in_domain'][0])

# Fetch eic.DISPLAY_NAME
no_name = norwegian_eic_codes[0].DISPLAY_NAME


NO


In [165]:
# Create dict of 'out_domain': 'in_domain'
domain_dict = df.set_index('out_domain')['in_domain'].to_dict()
table_data = []

for out_domain, in_domain in domain_dict.items():
    display_name_out = next((eic.DISPLAY_NAME for eic in all_eic_countries if eic.EIC_CODE == out_domain), 'Unknown')
    display_name_in = next((eic.DISPLAY_NAME for eic in all_eic_countries if eic.EIC_CODE == in_domain), 'Unknown')

    table_name = f"import_{display_name_out.lower()}_{display_name_in.lower()}" if display_name_in == 'NO' else f"export_{display_name_out.lower()}_{display_name_in.lower()}"
    table_data.append([out_domain, in_domain, display_name_out, display_name_in, table_name])

In [174]:
db_keyspace = "ind320"
table_df = pd.DataFrame(table_data, columns=['domain_out', 'domain_in', 'display_name_out', 'display_name_in', 'table_name'])

# Set table_df row index as the id
table_df['id'] = table_df.index

display(table_df)

spark_df = spark.createDataFrame(table_df)
spark_df.show()
spark_df.write.format("org.apache.spark.sql.cassandra").options(table="entsoe_tables", keyspace=db_keyspace).mode("append").save()

,domain_out,domain_in,display_name_out,display_name_in,table_name,id
0,10Y1001A1001A65H,10YNO-0--------C,DK,NO,import_dk_no,0
1,10Y1001A1001A83F,10YNO-0--------C,DE,NO,import_de_no,1
2,10YFI-1--------U,10YNO-0--------C,FI,NO,import_fi_no,2
3,10YGB----------A,10YNO-0--------C,GB,NO,import_gb_no,3
4,10YNL----------L,10YNO-0--------C,NL,NO,import_nl_no,4
5,10YSE-1--------K,10YNO-0--------C,SE,NO,import_se_no,5


+----------------+----------------+----------------+---------------+------------+---+
|      domain_out|       domain_in|display_name_out|display_name_in|  table_name| id|
+----------------+----------------+----------------+---------------+------------+---+
|10Y1001A1001A65H|10YNO-0--------C|              DK|             NO|import_dk_no|  0|
|10Y1001A1001A83F|10YNO-0--------C|              DE|             NO|import_de_no|  1|
|10YFI-1--------U|10YNO-0--------C|              FI|             NO|import_fi_no|  2|
|10YGB----------A|10YNO-0--------C|              GB|             NO|import_gb_no|  3|
|10YNL----------L|10YNO-0--------C|              NL|             NO|import_nl_no|  4|
|10YSE-1--------K|10YNO-0--------C|              SE|             NO|import_se_no|  5|
+----------------+----------------+----------------+---------------+------------+---+



In [172]:
# Connecting to Cassandra
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()

db_keyspace = "ind320"
db_table = "entsoe_tables"

session.execute(f"CREATE KEYSPACE IF NOT EXISTS {db_keyspace} WITH REPLICATION = {{ 'class': 'SimpleStrategy', 'replication_factor': 1 }};")
session.set_keyspace(db_keyspace)

session.execute(f"DROP TABLE IF EXISTS {db_table};")

session.execute(f"""CREATE TABLE IF NOT EXISTS {db_table} (
    id INT PRIMARY KEY,
    domain_out TEXT,
    domain_in TEXT,
    display_name_out TEXT,
    display_name_in TEXT,
    table_name TEXT
);""")

In [175]:
# Check if table populated
spark_df = spark.read.format("org.apache.spark.sql.cassandra").options(table=db_table, keyspace=db_keyspace).load()
spark_df.show()

+---+---------------+----------------+----------------+----------------+------------+
| id|display_name_in|display_name_out|       domain_in|      domain_out|  table_name|
+---+---------------+----------------+----------------+----------------+------------+
|  5|             NO|              SE|10YNO-0--------C|10YSE-1--------K|import_se_no|
|  1|             NO|              DE|10YNO-0--------C|10Y1001A1001A83F|import_de_no|
|  3|             NO|              GB|10YNO-0--------C|10YGB----------A|import_gb_no|
|  2|             NO|              FI|10YNO-0--------C|10YFI-1--------U|import_fi_no|
|  4|             NO|              NL|10YNO-0--------C|10YNL----------L|import_nl_no|
|  0|             NO|              DK|10YNO-0--------C|10Y1001A1001A65H|import_dk_no|
+---+---------------+----------------+----------------+----------------+------------+



In [176]:
df['id'] = df.index

In [178]:
display(df)

,in_domain,out_domain,timestamp,quantity,unit,resolution,business_type,id
0,10YNO-0--------C,10Y1001A1001A65H,2024-10-19 20:00:00+00:00,0.000000,MAW,PT60M,A66,0
1,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 08:00:00+00:00,22.410000,MAW,PT60M,A66,1
2,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 09:00:00+00:00,385.600000,MAW,PT60M,A66,2
3,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 10:00:00+00:00,664.130000,MAW,PT60M,A66,3
4,10YNO-0--------C,10Y1001A1001A65H,2024-10-20 11:00:00+00:00,748.760000,MAW,PT60M,A66,4
...,...,...,...,...,...,...,...,...
86466,10YNO-0--------C,10YSE-1--------K,2026-10-19 16:45:00+00:00,142.394402,MAW,PT15M,A66,86466
86467,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:00:00+00:00,187.579018,MAW,PT15M,A66,86467
86468,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:15:00+00:00,260.305923,MAW,PT15M,A66,86468
86469,10YNO-0--------C,10YSE-1--------K,2026-10-19 17:30:00+00:00,365.808594,MAW,PT15M,A66,86469


In [187]:
# for table_name in table_df['table_name']:
    
#     print(table_name)
#     db_keyspace = "ind320"

#     session.execute(f"DROP TABLE IF EXISTS {table_name};")

#     session.execute(f"""CREATE TABLE IF NOT EXISTS {table_name} (
#         id INT PRIMARY KEY,
#         domain_out TEXT,
#         domain_in TEXT,
#         timestamp TIMESTAMP,
#         quantity FLOAT,
#         unit TEXT,
#         resolution TEXT,
#         business_type TEXT
#     );""")
    
for table_name in table_df['table_name']:
    
    print(table_name)
    df_to_push = df[df['out_domain'] == (table_df[table_df['table_name'] == table_name]['domain_out'].iloc[0])].copy()
    # Rename 'out_domain' to 'domain_out' and 'in_domain' to 'domain_in'
    df_to_push.rename(columns={'out_domain': 'domain_out', 'in_domain': 'domain_in'}, inplace=True)
    #display(df_to_push.head())
    spark_df = spark.createDataFrame(df_to_push)
    spark_df.write.format("org.apache.spark.sql.cassandra").options(table=table_name, keyspace=db_keyspace).mode("append").save()

import_dk_no


import_de_no
import_fi_no
import_gb_no
import_nl_no
import_se_no


In [189]:
# Check if all tables exist
for table_name in table_df['table_name']:
    spark_df = spark.read.format("org.apache.spark.sql.cassandra").options(table=table_name, keyspace=db_keyspace).load()
    print(f"Data for table {table_name}:")
    spark_df.show(5)

Data for table import_dk_no:
+-----+-------------+----------------+----------------+---------+----------+-------------------+----+
|   id|business_type|       domain_in|      domain_out| quantity|resolution|          timestamp|unit|
+-----+-------------+----------------+----------------+---------+----------+-------------------+----+
|13462|          A66|10YNO-0--------C|10Y1001A1001A65H|1608.3392|     PT15M|2026-05-29 14:45:00| MAW|
| 8720|          A66|10YNO-0--------C|10Y1001A1001A65H|1604.7219|     PT15M|2026-02-22 01:15:00| MAW|
|14470|          A66|10YNO-0--------C|10Y1001A1001A65H|108.74322|     PT15M|2026-06-26 10:30:00| MAW|
| 9770|          A66|10YNO-0--------C|10Y1001A1001A65H|1605.0891|     PT15M|2026-03-13 10:15:00| MAW|
|18487|          A66|10YNO-0--------C|10Y1001A1001A65H|1117.9567|     PT15M|2026-10-10 04:15:00| MAW|
+-----+-------------+----------------+----------------+---------+----------+-------------------+----+
only showing top 5 rows

Data for table import_de_no: